# AWS SageMaker: Entrenamiento y Optimización de Modelos ML
## Laboratorio Consolidado - Prevención de Churn

Este notebook consolida el flujo completo de entrenamiento de un modelo de machine learning usando AWS SageMaker.

**Objetivo:** Predecir qué clientes tienen riesgo de abandonar el servicio (churn) para priorizar acciones de retención.

## 1️⃣ Contexto de Negocio

### El Problema
- **Necesidad:** Identificar clientes en riesgo de churn
- **Impacto:** Permitir acciones proactivas de retención
- **Métrica Principal:** F1 Score (balanceamos Precision y Recall)

### Conceptos Clave

| Término | Significado |
|---------|-------------|
| **Churn** | Abandono del cliente |
| **Target** | `churn_label` (0 = no churn, 1 = churn) |
| **Recall** | Capacidad de detectar clientes que realmente abandonarían |
| **Precision** | Proporción de alertas que son correctas |
| **F1 Score** | Media armónica entre Precision y Recall |

### Tipos de Datos del Modelo
- **Características Demográficas:** edad, país, tipo de plan
- **Comportamiento:** uso del servicio, soporte contactado
- **Financiero:** meses de suscripción, pagos atrasados

## 2️⃣ Arquitectura General

```
┌─────────────────────────────────────────────────────────────┐
│                      FLUJO DE ENTRENAMIENTO                  │
└─────────────────────────────────────────────────────────────┘

  DATOS CRUDOS (S3: raw/)
         │
         ↓
  LIMPIEZA Y PREPARACIÓN
         │
         ↓
  DATOS CURADOS (S3: curated/)
         │
         ↓
  ┌──────────────────────────────────┐
  │  SAGEMAKER FEATURE STORE         │
  ├──────────────────────────────────┤
  │ • Online Store (Low Latency)     │
  │ • Offline Store (S3 Histórico)   │
  └──────────────────────────────────┘
         │
         ↓
  PROCESSING JOB (Preparación Datasets)
  └─→ train.csv, validation.csv, test.csv
         │
         ├─────────────────┬──────────────┐
         ↓                 ↓              ↓
    BASELINE       HPO (Tuning)    AUTOPILOT (Opcional)
   TRAINING       TRAINING JOBS    AUTOMÁTICO
         │                 │              │
         └─────────────────┴──────────────┘
                    │
                    ↓
         EVALUACIÓN Y COMPARACIÓN
                    │
                    ↓
      MODELO REGISTRY (Gobernanza)
         PendingManualApproval
```

### Servicios AWS Utilizados
- **S3:** Almacenamiento de datos, código y artefactos
- **SageMaker Feature Store:** Repositorio de features con Online/Offline Store
- **AWS Glue & Athena:** Materializar datasets desde Feature Store
- **SageMaker Processing:** Ingesta batch y preparación reproducible
- **SageMaker Training:** Entrenamientos baseline
- **SageMaker HPO:** Búsqueda automática de hiperparámetros
- **SageMaker Model Registry:** Versionado y gobernanza de modelos
- **SageMaker Experiments:** Tracking de experimentos
- **SageMaker Pipelines:** Orquestación del flujo ML

## 3️⃣ Componentes Principales del Laboratorio

### 📊 PASO 03: SageMaker Feature Store

**Propósito:** Almacenar features de forma centralizada, versionada e histórica.

**Características:**
- **Record Identifier:** `customer_id` (identificador único)
- **Event Time:** `event_time` (timestamp para historial)
- **Online Store:** Lectura rápida por `customer_id` (inferencia en tiempo real)
- **Offline Store:** Historial en S3 (entrenamiento, auditoría, batch)

**Flujo:**
```
s3://bucket/curated/churn_features.csv
        ↓
Processing Job (feature_ingestion_entrypoint.py)
        ↓
PutRecord → Feature Store
        ├→ Online Store (para inference)
        └→ Offline Store (para training)
```

---

### 🔄 PASO 04: SageMaker Processing Jobs

**Propósito:** Preparar datasets reproducibles desde el Offline Store.

**Tareas:**
1. Consultar Feature Store Offline Store via Glue Data Catalog y Athena
2. One-hot encoding de variables categóricas
3. Normalización de features numéricas
4. Split en train.csv, validation.csv, test.csv
5. Guardar en S3 para consumo del Training Job

**Patrón Clave:** Feature Store → Athena → Processing → CSVs en S3 → Training

---

### 🚂 PASO 05: SageMaker Training Jobs

**Modelo Base:** `sklearn.linear_model.LogisticRegression`

**Pipeline:**
```
train.csv → StandardScaler → LogisticRegression → model.joblib
```

**Artefacto:** `s3://bucket/output/baseline/training-job-name/output/model.tar.gz`

**Métricas extraídas:**
- `validation:f1` (métrica objetivo)
- `validation:precision`
- `validation:recall`
- `validation:auc`

---

### 🔍 PASO 07: Hyperparameter Tuning (HPO)

**Objetivo:** Buscar hiperparámetros que maximicen `validation:f1`

**Hiperparámetros Optimizados:**

| Parámetro | Rango | Significado |
|-----------|-------|-------------|
| `C` | 0.01 - 10.0 (log) | Inverso de la fuerza de regularización |
| `max_iter` | 150 - 450 | Máximas iteraciones del solver |
| `class_weight` | {balanced, none} | Manejo del desbalanceo de clases |

**Flujo:**
```
Tuning Job
  └─ Trial 1 (C=0.01, max_iter=150)
  └─ Trial 2 (C=1.0, max_iter=300)
  └─ Trial 3 (C=10.0, max_iter=450)
  ...
  → Selecciona: Best Trial = máximo validation:f1
```

---

### 📦 PASO 09: SageMaker Model Registry

**Propósito:** Versionado, gobernanza y aprobación de modelos.

**Componentes:**
- **Model Package Group:** Contenedor de versiones (ej: `churn-model-package-group`)
- **Model Package:** Versión del modelo con artefacto, imagen, métricas
- **Approval Status:** `PendingManualApproval` → `Approved` → `Rejected`

**Estados en Studio:**
- `Deploy: Pending Approval` → Requiere aprobación manual antes de desplegar
- `Train: Complete` → Artefacto encontrado
- `Evaluate: Complete` → Métricas asociadas
- `Audit: Draft` → Model Card en borrador (documentación)

**Modelo Card:** Documentación Markdown con:
- Descripción del modelo
- Métricas y casos de uso
- Riesgos identificados
- Propietarios y contexto

## 4️⃣ Configuración y Setup

### Prerrequisitos
```bash
# 1. Crear ambiente virtual
python -m venv .venv

# 2. Activar en Linux/Mac
source .venv/bin/activate

# 2. Activar en Windows PowerShell
.venv\Scripts\Activate.ps1

# 3. Instalar dependencias
pip install -r requirements.txt
```

### Variables de Entorno (.env)
```bash
# AWS
AWS_PROFILE=your-profile
AWS_REGION=us-east-1

# Project
PROJECT_NAME=ml-training-opt-lab
ENVIRONMENT=dev
S3_BUCKET_NAME=your-bucket-name

# Feature Store
FEATURE_GROUP_NAME=churn-customer-features
ENABLE_ONLINE_STORE=true
ENABLE_OFFLINE_STORE=true

# Compute
PROCESSING_INSTANCE_TYPE=ml.m5.xlarge
TRAINING_INSTANCE_TYPE=ml.m5.large
TRAINING_INSTANCE_COUNT=1

# HPO
HPO_MAX_JOBS=20
HPO_MAX_PARALLEL_JOBS=4
```

## 5️⃣ Ejecución del Laboratorio

### Opción 1: Ejecutar Todo de una vez
```bash
# Con Make
make all-cloud

# O con Python
python -m src.lab_runner all

# O con Bash
bash scripts/lab.sh all

# O con PowerShell (Windows)
scripts\run_all_cloud.ps1
```

### Opción 2: Ejecutar por Pasos
```bash
make lab-00-context          # Contexto de negocio
make lab-01-aws-setup        # Setup de AWS
make lab-02-training-data    # Generar datos sintéticos
make lab-03-feature-store    # Crear Feature Store e ingestar
make lab-04-processing       # Preparar datasets (train/val/test)
make lab-05-training         # Entrenar baseline
make lab-06-evaluation       # Evaluar métricas
make lab-07-hpo              # Hyperparameter tuning
make lab-08-experiments      # Tracking con Experiments
make lab-09-model-registry   # Registrar modelo
make lab-10-pipeline         # Crear SageMaker Pipeline
make lab-11-cost             # Análisis de costos
make lab-12-cleanup          # Limpiar recursos
```

### Qué hace `make all-cloud`

```mermaid
graph LR
    A["Desplegar Infraestructura"] --> B["Generar Datos Sintéticos"]
    B --> C["Organizar S3: raw/ → cleaned/ → curated/"]
    C --> D["Crear Feature Store"]
    D --> E["Processing Job: Ingestar Features"]
    E --> F["Processing Job: Preparar Datasets"]
    F --> G["Entrenar Baseline"]
    G --> H["Ejecutar HPO"]
    H --> I["Evaluar y Comparar"]
    I --> J["Registrar en Model Registry"]
    J --> K["Generar Reportes"]
    K --> L["Validar Recursos"]
```

## 6️⃣ Código Principal del Proyecto

### Estructura de Carpetas
```
3_ML-Model-Training-Optimization/
├── src/
│   ├── config.py              # Configuración centralizada
│   ├── aws_clients.py         # Clientes de AWS
│   ├── create_feature_group.py # Feature Store setup
│   ├── ingest_features.py     # Ingesta de features
│   ├── create_pipeline.py     # Crea Processing para datasets
│   ├── train.py               # Entrenamiento del modelo
│   ├── create_hpo_pipeline.py # Hyperparameter tuning
│   ├── evaluate_model.py      # Evaluación y métricas
│   ├── approve_model.py       # Aprobación manual
│   └── ...
├── processing/
│   ├── feature_ingestion_entrypoint.py
│   ├── processing_entrypoint.py
│   ├── evaluation_entrypoint.py
│   └── utils.py
├── training/
│   └── train.py               # Script que se ejecuta en el container
├── scripts/
│   └── [Wrappers bash/ps1]
├── lab/
│   └── [Documentación de cada paso]
└── tests/
    └── [Tests unitarios]
```

## 7️⃣ Configuración Centralizada (config.py)

```python
from dataclasses import dataclass
import os

@dataclass(frozen=True)
class AppConfig:
    # AWS
    aws_profile: str | None
    aws_region: str
    
    # Proyecto
    project_name: str
    environment: str
    resource_prefix: str
    stack_name: str
    
    # S3 y IAM
    s3_bucket_name: str
    sagemaker_execution_role_arn: str
    
    # Feature Store
    feature_group_name: str
    enable_online_store: bool
    enable_offline_store: bool
    
    # Compute
    processing_instance_type: str
    training_instance_type: str
    processing_instance_count: int
    training_instance_count: int
    
    # HPO
    hpo_max_jobs: int
    hpo_max_parallel_jobs: int
    
    # Gobernanza
    model_package_group_name: str
    delete_feature_group_on_cleanup: bool
    delete_s3_objects_on_cleanup: bool
```

**Acceso desde cualquier módulo:**
```python
from src.config import load_env, AppConfig, get_app_config

load_env()  # Carga .env
config = get_app_config()  # Obtiene configuración
print(config.s3_bucket_name)  # Accede a valores
```

## 8️⃣ Flujo Detallado de Training

### A. Preparación de Datos (Processing Job)
```python
# processing/processing_entrypoint.py
def main():
    # 1. Leer del Offline Store
    df = spark.read.parquet(offline_store_path)
    
    # 2. Feature Engineering
    df = apply_feature_engineering(df)  # One-hot encoding, etc.
    
    # 3. Split de datos
    train_df = df[df.split == 'train']
    validation_df = df[df.split == 'validation']
    test_df = df[df.split == 'test']
    
    # 4. Guardar en S3
    train_df.to_csv('s3://bucket/input/train/train.csv')
    validation_df.to_csv('s3://bucket/input/validation/validation.csv')
    test_df.to_csv('s3://bucket/input/test/test.csv')
```

### B. Entrenamiento Base (Training Job)
```python
# training/train.py (ejecutado en el container SageMaker)
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

def main():
    # 1. Cargar datos
    train = pd.read_csv('/opt/ml/input/data/train/train.csv')
    validation = pd.read_csv('/opt/ml/input/data/validation/validation.csv')
    
    X_train = train.drop('churn_label', axis=1)
    y_train = train['churn_label']
    X_val = validation.drop('churn_label', axis=1)
    y_val = validation['churn_label']
    
    # 2. Crear pipeline
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=200))
    ])
    
    # 3. Entrenar
    model.fit(X_train, y_train)
    
    # 4. Evaluar
    f1 = f1_score(y_val, model.predict(X_val))
    print(f'validation:f1={f1}')
    
    # 5. Guardar
    joblib.dump(model, '/opt/ml/model/model.joblib')
```

### C. Hyperparameter Tuning
```python
# src/create_hpo_pipeline.py
from sagemaker.tuner import (
    IntegerParameter, ContinuousParameter, CategoricalParameter,
    HyperparameterTuner
)

tuner = HyperparameterTuner(
    estimator=sklearn_estimator,
    objective_metric_name='validation:f1',
    hyperparameter_ranges={
        'C': ContinuousParameter(0.01, 10.0, scaling_type='Log'),
        'max_iter': IntegerParameter(150, 450),
        'class_weight': CategoricalParameter(['balanced', 'none'])
    },
    max_jobs=20,
    max_parallel_jobs=4
)

tuner.fit(inputs)  # Ejecuta todos los trials
best_model = tuner.best_estimator  # Mejor trial
```

## 9️⃣ Métrica Principal: F1 Score

### ¿Por qué F1 Score?

En churn prediction, **Recall es crítico**: No queremos perder clientes en riesgo.

Pero **Precision también importa**: Contactar falsos positivos es costoso.

**F1 Score balancea ambas:**

$$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

### Interpretación
- **F1 = 0:** Modelo no funciona
- **F1 = 0.5:** Modelo mediocre
- **F1 = 0.8:** Modelo bueno
- **F1 = 1.0:** Modelo perfecto (raro en datos reales)

### Matriz de Confusión
```
                  Predicción
                Churn  No Churn
Real    Churn    TP      FN      → Recall = TP/(TP+FN)
Real  No Churn   FP      TN      → Precision = TP/(TP+FP)

TP = True Positive (detectamos churn correctamente)
FN = False Negative (no detectamos churn, error grave)
FP = False Positive (falsa alarma)
TN = True Negative (correctamente identificamos no-churn)
```

## 🔟 Gobernanza y Ciclo de Vida del Modelo

### Estado del Modelo en Model Registry
```
1. ENTRENAMIENTO
   └─ SageMaker Training Job
   └─ Genera: model.tar.gz → S3
   └─ Status: "Train: Complete"

2. EVALUACIÓN
   └─ Calcula: F1, Precision, Recall, AUC
   └─ Genera: evaluation_report.json
   └─ Status: "Evaluate: Complete"

3. REGISTRO
   └─ Registra en Model Registry
   └─ Status: "PendingManualApproval"
   └─ Genera: model_card.md (documentación)

4. APROBACIÓN MANUAL (Opcional)
   └─ Revisor humano valida
   └─ Status: "Approved" o "Rejected"

5. DESPLIEGUE (Siguiente lab)
   └─ Crear Endpoint
   └─ Realizar Inference en tiempo real o batch
```

### Model Card Generado Automáticamente
```markdown
# Churn Prediction Model Card

## Información del Modelo
- **Nombre:** Baseline Logistic Regression
- **Versión:** 2025-05-24
- **Propietario:** Data Science Team

## Caso de Uso
Identificar clientes con riesgo de churn en los próximos 30 días.

## Métricas
- F1 Score: 0.78
- Precision: 0.82
- Recall: 0.75
- AUC: 0.85

## Limitaciones
- Solo funciona con datos en formato esperado
- Requiere preprocesamiento (one-hot encoding)
- No generaliza a nuevos mercados sin reentrenamiento

## Riesgos
- Puede no detectar patrones nuevos de churn
- Sesgado hacia clientes históricos
```

## 1️⃣1️⃣ Análisis de Costos

### Desglose Típico de Costos

| Componente | Costo Típico | Notas |
|-----------|----------|-------|
| **Processing Jobs** | $0.50 - $2.00 | ml.m5.xlarge × 2 jobs |
| **Training Baseline** | $1.00 - $3.00 | ml.m5.large × 1 hour |
| **HPO (20 trials)** | $10.00 - $30.00 | 20 × ml.m5.large jobs |
| **Feature Store Ingesta** | $0.10 - $0.50 | Depende del volumen |
| **S3 Almacenamiento** | $0.023 / GB/mes | CSVs, artefactos, reports |
| **Athena Consultas** | $5 / TB escaneado | Feature Store Offline |
| **TOTAL** | $15 - $50 | Por ejecución completa |

### Optimizaciones de Costo
```python
# En config.py
HPO_MAX_JOBS = 10  # Menos trials = menos costo
PROCESSING_INSTANCE_TYPE = 'ml.m5.large'  # Instancia más pequeña
WAIT_FOR_JOBS = False  # Polling asincrónico

# Limpiar después
make lab-12-cleanup  # Elimina Feature Store, Datasets, etc.
```

## 1️⃣2️⃣ Próximos Pasos (Siguientes Labs)

### Lab 04: Despliegue de Modelos
- Crear endpoint SageMaker para inference en tiempo real
- Despliegue con canary deployment
- Monitoreo de drift en producción

### Lab 05: MLOps
- Automatizar el flujo completo con CI/CD
- SageMaker Pipelines como DAG orquestado
- Lambda para triggering automático
- EventBridge para eventos

### Evolución del Modelo
```
Baseline (Logistic Regression)
  ↓
XGBoost / Random Forest (HPO)
  ↓
Deep Learning (Neural Network)
  ↓
Ensemble de modelos
  ↓
Reentrenamiento automático mensual
```

## 📚 Resumen: Pasos Clave para Recordar

### 1. **Feature Store** = Repositorio centralizado
   - Online Store para inferencia rápida
   - Offline Store para entrenamiento reproducible

### 2. **Processing Job** = Preparación reproducible
   - Carga datos desde Feature Store
   - Aplica transformaciones (one-hot, scaling)
   - Genera CSVs listos para training

### 3. **Training Job** = Entrena el modelo
   - Lee CSVs de S3
   - Ajusta pipeline (Scaler + Classifier)
   - Guarda model.tar.gz en S3

### 4. **HPO (Tuning)** = Busca mejores hiperparámetros
   - Ejecuta múltiples Training Jobs
   - Maximiza métrica objetivo (F1 Score)
   - Elige el mejor trial

### 5. **Model Registry** = Versiona y aprueba
   - Registra modelo con metadatos
   - Requiere aprobación manual
   - Genera Model Card documentado

### 6. **Gobernanza** = Auditoría completa
   - Lineage: qué datos alimentaron el modelo
   - Métricas: performance en cada dataset
   - Aprobadores: quién validó el modelo
   - Timestamps: cuándo fue entrenado/aprobado